# Python in the Smoke 💨 🐍 💨
## Applied notebook: audio analysis with the `librosa` library

---

**Repo:** [`python-in-the-smoke`](https://markgotham.github.io/python-in-the-smoke)

**By:** Mark Gotham

**Licence:** MIT

**Colour and symbol key:** [Itemised and explained here](https://markgotham.github.io/python-in-the-smoke/#note-to-students)

**Assumed knowledge:**

This notebook builds on the foundational Part I notebooks (a to g).
No other applied notebook is required, and **no knowledge of audio or music technology is assumed**: sound is explained as we go.
Each idea in the table below has appeared before, and now returns in a new setting.

| Part I | Assumed knowledge | How it appears here |
|---|---|---|
| a | numbers, text (strings), f-strings | Frequencies and note names; formatted tables of output |
| b | lists, dictionaries | Lists of notes, tempi and errors |
| c | Booleans, conditionals | Deciding whether an estimate is correct |
| d | loops | Looping over notes, noise levels and tempi |
| e | functions | `make_tone`, `cents_error`, `match_times` |
| f | packages | Importing `librosa`, `numpy` and `matplotlib` |
| g | files and paths | Only in Part 5, and handled by `librosa` |

**New here, and explained where they appear:** `numpy` **arrays** (Part 1), **plots** (Part 1), **random numbers with a seed** (Part 3) and **tolerance matching** of times (Part 2).

---
## 🟢 Information: Why this topic, and why this design?

**The cultural question.**
Sound is data too!
From production-quality recordings of professional performances and broadcasts to "found sounds" and archives,
it's all rich in culture!
Here we focus on some basic musical audio analysis,
estimating a pitch, a tempo or the moments when notes begin.
This kind of analysis can enrich a collection's catalogue without requiring hours of manual annotation.
But how far can we trust those estimates?
As in the [NLP notebook](./text_nlp.ipynb),
we ask what a tool can do,
and how to treat its output as a **claim to test**, not a fact.

**The advantage of audio.**
We can *make* sounds whose pitch and tempo we know in advance, so we have the right answer to compare with.

**The plan in 5 acts (parts).**
1. **Part 1:** sound as a list of numbers, and a pitch tool that works.
2. **Part 2:** a way to measure what "works", against a known answer.
3. **Part 3:** conditions that break it: noise, chords, fast tempi and unusual tuning.
4. **Part 4:** patching with prior knowledge, and the risk of doing so.
5. **Part 5:** a blueprint for real recordings, where the right answer is not known.

📝 **Parts 1 to 4 use synthetic sound**, generated by code.
This is very safely free of copyright questions, and the clocks, chimes and buskers in it are invented.
Real recordings come in Part 5, which you download yourself.

📝 **Everything you hear is also shown** as numbers or plots, so nothing in the notebook depends on being able to hear it.

---
## 🟢 Information: Set up

We need three libraries:
- `librosa`: audio analysis.
- `numpy`: the library for arrays of numbers, which `librosa` is built on.
- `matplotlib`: plotting.

Run the cell below once. On Colab, `numpy` and `matplotlib` are usually installed already.

📝 We ask for `librosa` version 1.0, because the outputs quoted in the solutions come from that version.
Another version may give different results.
If the install fails on a very new Python, install the latest version instead and expect differences.

In [ ]:
%pip install "librosa>=1.0,<1.1" matplotlib

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio

In [ ]:
print("librosa", librosa.__version__, "| numpy", np.__version__)

📝 **Record your versions.** Results depend on the library versions, so write them down beside any findings.

📝 **The first analysis call can be slow** (a few seconds), because `librosa` prepares some of its code on first use.
Later calls are quick, so do not worry if the first one seems to hang.

---

# Part 1: Sound as numbers

## 🟢 Information: What is a sound, to a computer?

Sound is a pattern of air pressure that rises and falls, many times per second.
A computer stores it by measuring that pressure very often and writing the measurements down in a long list of numbers.
- Each measurement is a **sample**.
- The number of samples per second is the **sample rate** (`sr`). We use 22,050 samples per second here.
- The number of up-and-down cycles per second is the **frequency**, measured in **hertz (Hz)**.
  The A above middle C is 440 Hz, and we hear frequency as **pitch**.

`librosa.tone(440, sr=sr, duration=2)` makes two seconds of a pure 440 Hz tone for us.

In [ ]:
sr = 22050  # samples per second

y = librosa.tone(440, sr=sr, duration=2)

print(type(y))
print(len(y), "samples")
print(y[:5])

## 🟢 New topic: `numpy` arrays

The result is not a normal list but a **`numpy` array**: a list of numbers designed for fast maths.
Arrays work like lists in some ways (`len`, indexing, slices such as `y[:5]`), and differently in one big way: **arithmetic applies to every item at once**.
`numpy` (imported as `np`) also has functions such as `np.max` and `np.median`.

In [ ]:
print(len(y))           # how many samples
print(y[:5])            # the first five
print(y.max(), y.min()) # the highest and lowest values
print((y * 0.5)[:5])    # halve every sample at once
print(len(y) / sr, "seconds")

## 🟢 Hear it, and see it

`Audio(y, rate=sr)` makes a player in the notebook.
For the picture we use a **plot**: `plt.plot(...)` draws a line through the numbers, with labels added around it, and `plt.show()` displays it.
We plot only the first 200 samples (less than a hundredth of a second), so we can see individual waves.

In [ ]:
Audio(y, rate=sr)

In [ ]:
plt.plot(y[:200])
plt.xlabel("sample number")
plt.ylabel("amplitude")
plt.title("The first 200 samples of a 440 Hz tone")
plt.show()

---
### 🟣 Core Exercise: How many waves?

The tone has 440 waves per second, and the plot shows the first 200 samples out of 22,050 per second.
1. Work out how many complete waves you expect to see in the first 200 samples.
2. Count the peaks in the plot above. Do they agree?
3. Make a plot of the first 100 samples. How many waves do you expect now?

*Your answer here.*

---
---
---

### ✅ Solution: How many waves?

In [ ]:
# 440 waves per second, 200 samples out of 22050 per second:
print(440 * 200 / sr)

plt.plot(y[:100])
plt.xlabel("sample number")
plt.ylabel("amplitude")
plt.title("The first 100 samples")
plt.show()

440 × 200 ÷ 22,050 is about 4, so we expect about four waves, and the plot has four peaks.
For 100 samples we expect about two.
The sample rate links the two worlds: *samples* in the array, and *seconds* in the physical sound.

## 🟢 Information: Estimating the pitch

`librosa.yin` estimates the pitch of a sound, as a series of estimates, one for each short stretch (frame) of the sound.
We give it a **search range**: `fmin` and `fmax`, the lowest and highest frequencies it should consider.
The median of the estimates is a good single summary.
`librosa.hz_to_note` turns a frequency into a note name.

In [ ]:
f0 = librosa.yin(y, fmin=65, fmax=2000, sr=sr)

print(len(f0), "estimates; the first three:", f0[:3])

pitch = np.median(f0)
print(round(float(pitch), 1), "Hz, which is", librosa.hz_to_note(pitch))

We asked for 440 Hz and the tool says about 440.4 Hz, the note A4.
That is the tool working as intended.
The rest of the notebook asks how far that goes.

---
### 🟣 Core Exercise: A different note

1. Choose a note name (such as `"E4"`) and use `librosa.note_to_hz` to find its frequency.
2. Make one second of tone at that frequency.
3. Estimate its pitch as above, and print the estimate in Hz and as a note name.
4. Compare with the frequency you started with.

*Your answer here.*

---
---
---

### ✅ Solution: A different note

In [ ]:
true_freq = librosa.note_to_hz("E4")
print("true frequency:", round(float(true_freq), 2), "Hz")

y_e = librosa.tone(true_freq, sr=sr, duration=1)
f0_e = librosa.yin(y_e, fmin=65, fmax=2000, sr=sr)
estimate = np.median(f0_e)
print("estimate:", round(float(estimate), 2), "Hz,", librosa.hz_to_note(estimate))

The estimate is within a couple of cents of the truth: about 330.0 Hz against 329.6 Hz, and `hz_to_note` gives `E4`.
Any note in this range should behave similarly.
Again, that is the tool working as intended, and the next part asks how to measure it properly.

---

# Part 2: Measuring what "works" means in practice

## 🟢 Information: Compare with a known answer

Because we *made* the sound, we know the right answer.
We need two things: a way to build more interesting sounds, and a way to measure the error.

**Building sound.**
A real instrument does not make a pure tone.
It makes the main frequency plus **harmonics**: multiples of it (2×, 3×, ...) that are quieter.
`make_tone` adds a chosen number of harmonics, and `make_melody` joins several notes end to end (`np.concatenate` sticks arrays together).

In [ ]:
def make_tone(freq, harmonics=1, seconds=0.5):
    t = np.arange(int(sr * seconds)) / sr  # the time of each sample, in seconds
    wave = np.zeros(len(t))
    for h in range(1, harmonics + 1):
        wave = wave + np.sin(2 * np.pi * freq * h * t) / h
    return wave / np.max(np.abs(wave))


def make_melody(note_names, harmonics=1, seconds=0.5):
    pieces = []
    for name in note_names:
        pieces.append(make_tone(librosa.note_to_hz(name), harmonics, seconds))
    return np.concatenate(pieces)

**Measuring pitch error.**
Frequencies are best compared on a log scale, because we hear pitch that way (doubling the frequency is always an octave).
The unit is the **cent**: 100 cents is a semitone (neighbouring piano keys) and 1,200 cents is an octave.
- `cents_error(estimate, truth)` is positive if the estimate is too high, negative if too low.
- We call an estimate **correct** if it is within 50 cents of the truth: less than half a semitone, so the nearest note name is still right.

That tolerance is a **decision**: a stricter one (10 cents) or looser one would change the score.

In [ ]:
TOLERANCE_CENTS = 50


def cents_error(estimate, truth):
    return 1200 * np.log2(estimate / truth)


def estimate_notes(y, n_notes, fmin=65, fmax=2000):
    """Estimate one pitch for each of n_notes equal-length notes."""
    f0 = librosa.yin(y, fmin=fmin, fmax=fmax, sr=sr)
    part = len(f0) // n_notes
    estimates = []
    for i in range(n_notes):
        middle = f0[i * part + 3:(i + 1) * part - 3]  # skip the edges of each note
        estimates.append(np.median(middle))
    return estimates

Here is a worked example: a four-note chime of our own invention.
We estimate each note and compare with the truth.

In [ ]:
chime_notes = ["E4", "C4", "D4", "G3"]

chime = make_melody(chime_notes)
Audio(chime, rate=sr)

In [ ]:
true_freqs = [librosa.note_to_hz(name) for name in chime_notes]
estimates = estimate_notes(chime, len(chime_notes))

n_correct = 0
for i in range(len(chime_notes)):
    error = cents_error(estimates[i], true_freqs[i])
    correct = abs(error) <= TOLERANCE_CENTS
    if correct:
        n_correct += 1
    print(f"{chime_notes[i]}: true {true_freqs[i]:6.1f} Hz, estimate {estimates[i]:6.1f} Hz, "
          f"error {error:+5.1f} cents, correct: {correct}")

print(n_correct, "of", len(chime_notes), "correct")

All four notes are within a few cents.
A perfect score on a clean sound is encouraging but says little: the interesting question is where the score stops being perfect.

---
### 🟣 Core Exercise: Your own melody

1. Choose a melody of five notes between `"C4"` and `"G4"` (as a list of note names).
2. Build it with `make_melody`, and estimate each note with `estimate_notes`.
3. Print the error in cents for each note, and how many are correct.
4. Try `harmonics=5` in `make_melody`. Does it change the result?

*Your answer here.*

---
---
---

### ✅ Solution: Your own melody

In [ ]:
my_notes = ["C4", "D4", "E4", "F4", "G4"]

for harmonics in [1, 5]:
    melody = make_melody(my_notes, harmonics=harmonics)
    estimates = estimate_notes(melody, len(my_notes))
    errors = []
    for i in range(len(my_notes)):
        errors.append(round(float(cents_error(estimates[i], librosa.note_to_hz(my_notes[i]))), 1))
    print("harmonics", harmonics, "| errors in cents:", errors)

With `librosa` 1.0.1, the pure-tone melody has errors under 2 cents on every note, and the five-harmonic version under 1 cent.
So both score 5 of 5 against our tolerance of 50 cents, and the richer sound is, if anything, slightly easier for the tool.
On a clean sound, that is hardly a test.

## 🟢 Information: Measuring times

Pitch is one kind of claim (in the frequency domain).
Another is **when** things happen (the time domain).
E.g., when does a drum hit begin (**onsets**).
To test it we need a sound with known event times:
let's use a **click track** at a chosen tempo, in beats per minute (BPM).

Here we cannot ask for exact equality:
a detected time of 0.502 seconds should count as matching a true 0.5 seconds.
So we choose a **window** (a tolerance) in seconds, and use two measures, as for the entities in the NLP notebook:
- **Precision:** of the times the tool found, what fraction are near a true time?
- **Recall:** of the true times, what fraction did the tool find?

The `near` function in the next cell checks whether a time is within the window of any time in a list.

In [ ]:
def make_clicks(bpm, seconds=12):
    times = np.arange(0, seconds, 60 / bpm)  # the true times of the clicks
    y = librosa.clicks(times=times, sr=sr, length=int(sr * seconds))
    return y, times


def near(time, others, window):
    for other in others:
        if abs(other - time) <= window:
            return True
    return False


def match_times(detected, true_times, window=0.05):
    found = 0
    for d in detected:
        if near(d, true_times, window):
            found += 1
    recovered = 0
    for t in true_times:
        if near(t, detected, window):
            recovered += 1
    return {"detected": len(detected), "true": len(true_times),
            "precision": found / len(detected), "recall": recovered / len(true_times)}

In [ ]:
clicks_120, true_times = make_clicks(120)
Audio(clicks_120, rate=sr)

In [ ]:
detected = librosa.onset.onset_detect(y=clicks_120, sr=sr, units="time")
match_times(detected, true_times, window=0.05)

---
### 🟣 Core Exercise: Change the window

The window of 0.05 seconds (50 ms) is a decision.
1. Run `match_times` again on the same detected times with a very small window (0.001 seconds) and a very large one (0.3 seconds).
2. Report precision and recall for each, and say in one sentence which one you would trust, and why.

*Your sentence here.*

---
---
---

### ✅ Solution: Change the window

In [ ]:
for window in [0.001, 0.05, 0.3]:
    print(window, match_times(detected, true_times, window=window))

- **0.001 s:** precision and recall both fall to 0. The tool's times are not that exact: the first detected onset is at about 0.51 s for a true 0.5 s, because the analysis works in frames of about 0.023 s. A window smaller than that makes a good tool look useless.
- **0.05 s and 0.3 s:** identical here (precision 1.0, recall 0.96). The one missed time is the click at 0 s, which the tool does not report at the very start of the sound, whatever the window.

We would trust 0.05 s: it is larger than the tool's own timing resolution, and well below half the gap between clicks (0.25 s at 120 BPM).
A window that is too wide starts to match the *wrong* event, which flatters the score, so the tolerance is a decision to state alongside any result.

---

# Part 3: Where it breaks

## 🟢 Information: Noise

Real recordings are never clean.
We add **random noise** to the chime at a chosen signal-to-noise ratio (SNR), in decibels (dB):
- 10 dB: the noise is much weaker than the sound.
- 0 dB: the noise is as strong as the sound.
- -5 dB: the noise is stronger than the sound.

📝 **New topic: random numbers with a seed.**
`np.random.default_rng(0)` makes a random-number generator.
The `0` is its **seed**: the same seed always gives the same "random" numbers, so everyone gets the same results.
Change the seed and you get a different, equally valid, noise.

In [ ]:
def add_noise(y, snr_db, seed=0):

    rng = np.random.default_rng(seed)
    noise = rng.normal(size=len(y))
    noise = noise * np.sqrt(np.mean(y ** 2) / (10 ** (snr_db / 10) * np.mean(noise ** 2)))
    return y + noise


noisy_chime = add_noise(chime, snr_db=10)
Audio(noisy_chime, rate=sr)

---
### 🟣 Core Exercise: Predict, then check

*Before* running anything, predict in the Markdown cell below:
- the noise level at which the **pure-tone** chime (`harmonics=1`) first goes wrong by more than 50 cents
- whether the **harmonic** chime (`harmonics=5`) will be more robust or less robust than the pure one

Then, for each of the two kinds of chime and each noise level in `[None, 10, 0, -5]` (`None` means no noise), print the error of each note in cents, and check your predictions.

*Your predictions here.*

---
---
---

### ✅ Solution: Predict, then check

In [ ]:
for harmonics in [1, 5]:
    print("harmonics:", harmonics)
    clean = make_melody(chime_notes, harmonics=harmonics)
    for snr in [None, 10, 0, -5]:
        if snr is None:
            test = clean
        else:
            test = add_noise(clean, snr)
        estimates = estimate_notes(test, len(chime_notes))
        errors = []
        for i in range(len(chime_notes)):
            errors.append(round(float(cents_error(estimates[i], true_freqs[i]))))
        print(f"snr={str(snr):5}  errors of {errors} cents")

With `librosa` 1.0.1 and seed 0:
- `snr` of `None`, `10`, and `0` all yield close matches, albeing with the numbers creeping up at snr 0.
- For `snr=-5` the numbers jump with all cases wrong by an octave or more.
- The pure-tone version generally performs worse ... until `snr=-5`: richer sounds were not more robust once the noise was loud.
- The failures are not drifts but jumps of whole octaves (about 1,200 or 2,400 cents each),
- The results depend on the seed: try `add_noise(clean, snr, seed=1)` and different notes fail.

## 🟢 Information: Chords

`librosa.yin` assumes **one note at a time**.
What happens with a chord: three notes at once?
For comparison we use a different tool, `librosa.feature.chroma_cqt`, which measures how much energy there is at each of the twelve note names (C, C♯, D, ... B), whatever the octave.

In [ ]:
PITCH_CLASSES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]



def make_chord(note_names, harmonics=3, seconds=2):
    chord = np.zeros(int(sr * seconds))
    for name in note_names:
        chord = chord + make_tone(librosa.note_to_hz(name), harmonics, seconds)
    return chord / np.max(np.abs(chord))


def strongest_pitch_classes(y, how_many=3):
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)  # 12 rows, one per note name
    average = chroma.mean(axis=1)                    # the average over time
    order = np.argsort(average)                      # positions, weakest first
    strongest = []
    for position in order[-how_many:]:
        strongest.append(PITCH_CLASSES[position])
    return strongest[::-1]                           # strongest first


c_major = make_chord(["C4", "E4", "G4"])
Audio(c_major, rate=sr)

In [ ]:
f0 = librosa.yin(c_major, fmin=65, fmax=2000, sr=sr)
print("yin says:", round(float(np.median(f0)), 1), "Hz,", librosa.hz_to_note(np.median(f0)))
print("chroma says the strongest note names are:", strongest_pitch_classes(c_major))

`yin` returns about 65.6 Hz (`C2`), two octaves below the lowest note of the chord (`C4`),
and right at the bottom edge of our search range (`fmin=65`).
A result at the edge of the search range is a warning sign:
here it is an artefact of the tool's assumption (one note at a time), not information about the sound.
`chroma_cqt` returns `C`, `E` and `G`, the right note names, strongest first.

The two tools answer different questions:
- `yin` asks "what is the pitch of this single note?",
- and chroma asks "which note names are present?".

Choosing a tool is part of your method.

---
### 🟣 Core Exercise: Another chord

A minor is made of the notes A, C and E.
1. Predict what `yin` and `strongest_pitch_classes` will say for A minor (`["A3", "C4", "E4"]`).
2. Run both and compare with your prediction.

*Your prediction here.*

---
---
---

### ✅ Solution: Another chord

In [ ]:
a_minor = make_chord(["A3", "C4", "E4"])
f0 = librosa.yin(a_minor, fmin=65, fmax=2000, sr=sr)
print("yin says:", round(float(np.median(f0)), 1), "Hz,", librosa.hz_to_note(np.median(f0)))
print("chroma says:", strongest_pitch_classes(a_minor))

`yin` again returns about 65.9 Hz (`C2`), at the bottom edge of the range: the same artefact, and not the chord's root (A).
If you predicted that it would find the root, you were assuming it "understands" chords, which it does not.
Chroma gives `A`, `C` and `E`: correct.

## 🟢 Information: Tempo

`librosa.beat.beat_track` estimates a tempo (in BPM) and the beat times.
Our function `estimate_tempo` returns just the tempo.
We test it on click tracks of known tempo.

For tempo there is a known ambiguity: a tempo of 120 BPM can be felt as 60 BPM (every other beat) or 240 BPM.
There is more than one defensible "right" answer, so we compute **two** scores:
- **strict:** within 4% of the true tempo.
- **octave-tolerant:** within 4% of the true tempo, or of half or double it.

Both tolerances are decisions, neither is objectively "correct".

📝 Newer `librosa` versions return the tempo as a one-item array, which is why we wrap it in `float(np.atleast_1d(...)[0])`.

In [ ]:
def estimate_tempo(y, **kwargs):
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr, **kwargs)
    return float(np.atleast_1d(tempo)[0])


def tempo_correct(estimate, true, tolerance=0.04, octave=False):
    candidates = [true]
    if octave:
        candidates = [true, true / 2, true * 2]
    for candidate in candidates:
        if abs(estimate - candidate) / candidate <= tolerance:
            return True
    return False


for bpm in [60, 90, 120]:
    clicks, _ = make_clicks(bpm)
    estimate = estimate_tempo(clicks)
    print(f"true {bpm:3} BPM, estimate {estimate:6.1f}, strict: {tempo_correct(estimate, bpm)}, "
          f"octave-tolerant: {tempo_correct(estimate, bpm, octave=True)}")

---
### 🟣 Core Exercise: Where does tempo go wrong?

*Before* running anything, predict in the Markdown cell below: at which tempo, between 40 and 240 BPM, will the estimate first go wrong, and in what way?

Then run `estimate_tempo` on click tracks from 40 to 240 BPM in steps of 40, print the estimate with both scores, and compare with your prediction.

*Your prediction here.*

---
---
---

### ✅ Solution: Where does tempo go wrong?

In [ ]:
for bpm in [40, 80, 120, 160, 200, 240]:
    clicks, _ = make_clicks(bpm)
    estimate = estimate_tempo(clicks)
    print(f"true {bpm:3} BPM, estimate {estimate:6.1f}, strict: {str(tempo_correct(estimate, bpm)):5}, "
          f"octave-tolerant: {tempo_correct(estimate, bpm, octave=True)}")

From 40 to 160 BPM the estimates are within about 2% of the truth (for example, 117.5 for 120), so they are correct on both scores.
At 200 BPM the estimate is 99.4, and at 240 BPM it is 117.5: both are about **half** the true tempo.
They fail the strict score but pass the octave-tolerant one.

So the tool does not fail randomly: above about 160 BPM it settles on half the tempo.
Whether that counts as an error is a decision: a listener might well tap every other click of a very fast track.
The 4% tolerance is also a decision (117.5 for 120 is 2% out).

**A changing tempo.**
A single tempo number assumes the tempo is steady.
Here is a click track that **accelerates** from 80 to 160 BPM over fourteen seconds, such as a busker speeding up.

In [ ]:
def make_accelerating_clicks(start_bpm=80, end_bpm=160, seconds=14):
    times = [0.0]
    while times[-1] < seconds:
        progress = times[-1] / seconds
        bpm_now = start_bpm + (end_bpm - start_bpm) * progress
        times.append(times[-1] + 60 / bpm_now)
    y = librosa.clicks(times=times, sr=sr, length=int(sr * seconds))
    return y, times


accelerating, _ = make_accelerating_clicks()
print("estimated tempo:", round(estimate_tempo(accelerating), 1), "BPM, for a track that goes from 80 to 160")
Audio(accelerating, rate=sr)

The tool gives **one** number, about 143.6 BPM, for a sound that runs from 80 to 160: neither the start, the end, nor the average (120).

That is not the fault of the tool but the question: "what is the tempo?" has no single answer here.

A fairer test of a changing tempo would compare the *beat times* with `match_times`, not a single number.

## 🟢 Information: Tuning: a decision built into the tool

`librosa.hz_to_note` gives the nearest note name, and it assumes that A4 is 440 Hz and that the octave is divided into twelve equal steps ("equal temperament").
Those assumptions are **decisions built into the tool**, like the word list in a sentiment tool.
The note name can hide how far from the label a sound really is.

In [ ]:
y_430 = librosa.tone(430, sr=sr, duration=1)
pitch = np.median(librosa.yin(y_430, fmin=65, fmax=2000, sr=sr))
print(round(float(pitch), 1), "Hz is labelled", librosa.hz_to_note(pitch))
print("but it is", round(float(cents_error(pitch, 440))), "cents away from A4 at 440 Hz")

---
### ⭐ Optional Exercise: Older pitch standards

Orchestras have not always tuned A to 440 Hz.
Some period-instrument groups today tune A to about 415 Hz, close to one semitone lower.
1. Make a tone at 415 Hz and estimate its pitch.
2. What note name does `hz_to_note` give, and how many cents is it from A4 at 440 Hz?
3. In one sentence: what could go wrong if a catalogue stored these note names for a historical recording?

*Your sentence here.*

---
---
---

### ✅ Solution: Older pitch standards

In [ ]:
y_415 = librosa.tone(415, sr=sr, duration=1)
pitch = np.median(librosa.yin(y_415, fmin=65, fmax=2000, sr=sr))
print(round(float(pitch), 1), "Hz is labelled", librosa.hz_to_note(pitch))
print(round(float(cents_error(pitch, 440))), "cents from A4 at 440 Hz")

A 415 Hz tone is labelled `G♯4`, exactly one semitone (100 cents) below A4.
By the tool's rules the label is correct, but it would be misleading: an ensemble playing "A" at 415 Hz would be catalogued as G♯, and a piece in A major as G♯ major.
The numbers (the frequency in Hz) keep the truth, while the labels carry the tool's assumption.
Store the frequency, and the reference pitch you assumed alongside any note names.

## 🟢 Information: Naming the errors

As with entities, it helps to give errors names:
- **wrong octave:** the right note name, in the wrong octave (or double or half the tempo).
- **wrong note:** a pitch that is not in the sound, because the tool's assumption (one note at a time) was broken.
- **missed:** a real event that was not found (lower recall).
- **spurious:** something found where there is nothing (lower precision).
- **mislabelled:** the estimate is close, but the label (a note name) hides a difference.
- **wrong kind of answer:** a single number, where the truth changes.

---
### 🟣 Core Exercise: Sort the errors

Look back at the experiments in this part.
Give one example of each error type (if there is no example of a type, say so):
wrong octave, wrong note, missed, spurious, mislabelled, wrong kind of answer.

*Your examples here.*

---
---
---

### ✅ Solution: Sort the errors

| Type | Example from this part |
|---|---|
| **Wrong octave** | A chime note an octave low at 0 dB noise (-1205 cents); 200 BPM estimated as 99.4 |
| **Wrong note** | The chord: `yin` gives 65.6 Hz, which is not in the sound at all |
| **Missed** | The click at 0 s is not found (recall 0.96) |
| **Spurious** | None in these experiments (precision stayed at 1.0). Noise might produce some: you could test that |
| **Mislabelled** | 430 Hz labelled `A4` (38 cents flat); 415 Hz labelled `G♯4` |
| **Wrong kind of answer** | One tempo (143.6 BPM) for a track that accelerates from 80 to 160 |

Compare with the NLP notebook: wrong labels and misses appear in both, while octave errors and "wrong kind of answer" are new here.

---

# Part 4: Patching with prior knowledge

## 🟢 Information: Priors

If we know something about the sound in advance, we can tell the tool.
- For pitch: a narrower search range (`fmin` and `fmax`) rules out candidates that are too high or too low, such as an octave below the real note.
- For tempo: a starting guess (`start_bpm`).

Here is the harmonic chime at 0 dB noise, estimated first with a wide range, then with a narrow one around the notes we expect (200 to 600 Hz).

In [ ]:
harmonic_chime = make_melody(chime_notes, harmonics=5)
noisy = add_noise(harmonic_chime, snr_db=0)

for fmin, fmax in [(65, 2000), (200, 600)]:
    estimates = estimate_notes(noisy, len(chime_notes), fmin=fmin, fmax=fmax)
    errors = []
    for i in range(len(chime_notes)):
        errors.append(round(float(cents_error(estimates[i], true_freqs[i]))))
    print(f"range {fmin}-{fmax} Hz: errors in cents {errors}")

With the wide range, two of the four notes are an octave or more out (-1178 and -1903 cents, as in Part 3).
With the narrow range, all four are within 25 cents.
The prior removed the octave errors: it worked because we told the tool where to look.

And for tempo: `start_bpm` is the tool's starting guess.
Here is the effect of different guesses on two click tracks.

In [ ]:
for true_bpm, start in [(200, 120), (200, 180), (120, 120), (120, 60)]:
    clicks, _ = make_clicks(true_bpm)
    estimate = estimate_tempo(clicks, start_bpm=start)
    print(f"true {true_bpm} BPM, start_bpm={start:3}: estimate {estimate:6.1f}")

`start_bpm` pulls the estimate towards the guess:
- For the 200 BPM track, a guess of 120 gives 99.4 (half), but a guess of 180 gives 198.8 (correct).
- For the 120 BPM track, a guess of 120 gives 117.5 (fine), but a guess of 60 gives 60.1 (half).

A prior fixes an error only if it is roughly right, and a wrong one *causes* the error.
The guess is itself a decision: where would it come from (a known genre, a score marking, someone listening)?

---
### ⭐ Optional Exercise: When the prior is wrong

The narrow range 200 to 600 Hz suited the chime.
1. Predict what happens if we use the same range for a *higher* melody: `["C5", "D5", "E5", "F5", "G5"]`, which is from about 523 to 784 Hz.
2. Build it, estimate it with `fmin=200, fmax=600`, and print the errors in cents. Compare with the wide range (65 to 2000).

*Your prediction here.*

---
---
---

### ✅ Solution: When the prior is wrong

In [ ]:
high_notes = ["C5", "D5", "E5", "F5", "G5"]
high = make_melody(high_notes)

for fmin, fmax in [(65, 2000), (200, 600)]:
    estimates = estimate_notes(high, len(high_notes), fmin=fmin, fmax=fmax)
    errors = []
    for i in range(len(high_notes)):
        errors.append(round(float(cents_error(estimates[i], librosa.note_to_hz(high_notes[i])))))
    print(f"range {fmin}-{fmax} Hz: errors in cents {errors}")

With the wide range, the high melody is fine (all within 2 cents).
With the narrow range, the three notes above 600 Hz (`E5`, `F5`, `G5`) come out an octave low, about -1200 cents.
The prior that fixed the chime broke this melody: a prior is an assumption about the sound, and it fails silently when the sound does not match.
If we had not printed the errors, we might not have noticed.

⚠️ **Testing on the data we tuned on is circular.**
We chose 200 to 600 Hz *because* we knew where the chime's notes were, so of course it works on the chime.
A fair test needs a **new** sound that was not used to choose the setting, and the setting should be stated as an assumption: "this works for sounds between 200 and 600 Hz".

---

# Part 5: Real recordings, a blueprint

## 🟢 Information: Downloading a real recording

So far every sound was synthetic.
`librosa` includes a small collection of **example recordings** that it downloads the first time you ask for them and keeps in a local cache.
Nothing is stored in this repository, and the cells below fall back to a synthetic sound if the download fails.

`librosa.util.list_examples()` shows what is available.

In [ ]:
librosa.util.list_examples()

⚠️ **Before you use any recording:**
- **Licences.** Each example has its own licence and credit: look them up in the `librosa` documentation, and keep the credit with the sound.
- **Two layers of rights.** A piece of music and a *recording* of it are separate. Even if the composition is out of copyright, a particular recording usually is not.
- **Your own recordings.** Sites such as
[Wikimedia Commons](https://commons.wikimedia.org/wiki/Main_Page),
the [Internet Archive](https://archive.org)
and
[Freesound](https://freesound.org)
hold recordings under different licences, and some need an account or key to download.
Check each file's terms. APIs and downloading are taught elsewhere, so here we only apply them.
- **Do not commit** audio to a public repository unless you have checked that you may.
- **Record** the source, the licence, the version and the date, as you would for any source.

## 🟢 Information: A real tempo test

The `snare` example is described as "an accelerating beat on a snare drum".
We know what Part 3 suggests for such a sound, so it is a good first test.

The cell below tries the download and, if it fails, says so and uses our own synthetic accelerating clicks instead.

In [ ]:
try:
    real_y, real_sr = librosa.load(librosa.example("snare"))
    source = "the librosa example 'snare'"
except Exception as error:
    print("Could not get the example:", error)
    real_y, real_sr = make_accelerating_clicks()[0], sr
    source = "our synthetic accelerating clicks (download failed)"

print("Using", source)
print(round(len(real_y) / real_sr, 1), "seconds at", real_sr, "samples per second")

In [ ]:
Audio(real_y, rate=real_sr)


In [ ]:
tempo, beats = librosa.beat.beat_track(y=real_y, sr=real_sr)
print("estimated tempo:", round(float(np.atleast_1d(tempo)[0]), 1), "BPM")
print(len(beats), "beats found")

---
### ⭐ Optional Exercise: Make your own ground truth

For a real recording there is no answer key: **you** have to make one.
1. Listen to the recording above (or look at the plot of the first few seconds) and count the beats in the first 5 seconds, by tapping along.
2. Work out your own tempo: beats counted × 12 = beats per minute.
3. Compare with the tool's estimate and with the beat times found by `beat_track` (`librosa.frames_to_time(beats, sr=real_sr)`) for the first 5 seconds.
4. State your tolerance and decide: did the tool work?

*Your count, your tempo and your verdict here.*

---
---
---

### ✅ Solution: Make your own ground truth

In [ ]:
beat_times = librosa.frames_to_time(beats, sr=real_sr)
first_five = []
for t in beat_times:
    if t <= 5:
        first_five.append(round(float(t), 2))

print("beats the tool found in the first 5 seconds:", len(first_five))
print(first_five)
print("tempo from that count:", len(first_five) * 12, "BPM")

Your results depend on your recording and on what you counted, so there is no single answer.
Things to check: whether the tool's beats line up with what you hear; whether one number can describe a tempo that changes (Part 3); and whether your own count is reliable, since *you* are now the gold standard.
Record the recording's name and source, your tolerance, and the `librosa` version beside any claim.

---
## 🟢 Information: Where next?

The idea of this notebook was to treat a tool's output as a claim to test.
Here are three ways to take it further.

- **Other kinds of sound.** The `librosa` examples include speech (`libri1`), a robin and humpback whale song. Which of our estimates mean anything for those?
- **Other tuning systems.** `librosa` includes note-naming functions for other systems, such as `librosa.hz_to_svara_h` (Hindustani) and `librosa.hz_to_svara_c` (Carnatic). What assumptions does each make, and who decided them?
- **Archives.** Tempo, pitch and onset estimates could be metadata for a sound archive. The questions are the same as for text: Where did it come from? What was the tool built for? How would you check it?

### ⭐ A brief demo: the spectrogram

A **spectrogram** shows how the frequencies in a sound change over time: time runs left to right, frequency bottom to top, and brightness is loudness.
It is the picture that many audio tools use.
Here is our four-note chime: each note is a bright stripe at its own height, with fainter stripes above it (the harmonics).

In [ ]:
chime_h = make_melody(chime_notes, harmonics=5)
spectrogram = np.abs(librosa.stft(chime_h))
spectrogram_db = librosa.amplitude_to_db(spectrogram, ref=np.max)

fig, ax = plt.subplots()
image = librosa.display.specshow(spectrogram_db, sr=sr, x_axis="time", y_axis="log", ax=ax)
fig.colorbar(image, ax=ax, format="%+2.0f dB")
ax.set_title("Spectrogram of the four-note chime")
plt.show()

The four notes appear as four stripes at different heights, and each has the harmonics we added above it.
Everything the pitch tool did can be read, roughly, from this picture: that is a useful check on its output.

---
## ⚠️ Common Errors

| Symptom | What to check |
|---|---|
| `ModuleNotFoundError: No module named 'librosa'` | Run the install cell, then restart the kernel and re-run from the imports |
| The first analysis call seems to hang | It is preparing code on first use: wait a few seconds |
| The audio player is silent or missing | Check `rate=sr` and your volume, and click play. Some previewers (such as GitHub's) do not show players |
| `LibsndfileError` or `FileNotFoundError` when loading | The path does not exist: check the spelling and the working directory (Notebook g) |
| A `ParameterError` from `librosa` | A setting is not allowed: read the message, as it names the setting |
| The pitch is exactly an octave out | An **octave error**: narrow the search range (Part 4), but remember the risk |
| The tempo is half or double the truth | The tempo ambiguity (Part 3): decide whether to accept it |
| A download fails | Check your internet connection. The notebook falls back to synthetic sound |
| Different results from someone else | Compare `librosa` and `numpy` versions (printed in the setup) and the random seed |

**The cells below fail on purpose.**
Read each message, work out the cause, then compare with the explanation underneath.

### ⚠️ Deliberate error 1: a tone with no length

In [ ]:
librosa.tone(440)

`ParameterError: either "length" or "duration" must be provided`: `librosa.tone` needs to know how long the tone should be.
Add `duration=2` (seconds) or `length=` (a number of samples).
`librosa` raises its own `ParameterError` when a setting is missing or not allowed, and the message names the problem.

### ⚠️ Deliberate error 2: a file that does not exist

In [ ]:
librosa.load("no_such_recording.wav")

`LibsndfileError: Error opening 'no_such_recording.wav'` (an older version may show a different error type, such as `FileNotFoundError`): there is no file at that path.
Check the spelling, the folder and the working directory (Notebook g).

### ⚠️ Deliberate error 3: a search range the tool cannot use

In [ ]:
librosa.yin(y, fmin=10, fmax=2000, sr=sr)

`ParameterError: fmin=10.000 is too small for frame_length=2048 ...`: to detect a very low frequency, the tool needs a stretch of sound (a frame) long enough to hold a full wave.
Raise `fmin`, or lengthen the frame with `frame_length`: the message even suggests values.
It is a reminder that the search range of Part 4 is not free to choose.

---
## 🟢 Information: Summary

| To ... | Write ... |
|---|---|
| Make a pure tone | `librosa.tone(freq, sr=sr, duration=2)` |
| Hear and see a sound | `Audio(y, rate=sr)`; `plt.plot(y[:200])` |
| Convert between names and frequencies | `librosa.note_to_hz("A4")`; `librosa.hz_to_note(440)` |
| Estimate pitch | `librosa.yin(y, fmin=..., fmax=..., sr=sr)`, then `np.median(...)` |
| Compare two pitches | cents: `1200 * np.log2(estimate / truth)` (100 cents = a semitone) |
| Find onsets and beats | `librosa.onset.onset_detect(...)`; `librosa.beat.beat_track(...)` |
| Score times | precision and recall within a window (a tolerance you choose) |
| Add reproducible noise | `np.random.default_rng(seed).normal(size=...)` |
| Note names for a chord | `librosa.feature.chroma_cqt(...)` |
| Load a real recording | `librosa.load(librosa.example("snare"))` |

A tool's output is a claim to test.
Compare it with a known answer on a sound you built, then with your own answer on a real one, state your tolerances, and record the versions you used.
A tool that works on a clean tone can still fail on noise, chords, fast or changing tempi, and unusual tuning.

**Final reflection:**
- Which of the tool's assumptions (one note at a time, steady tempo, A = 440 Hz) matter for the sounds *you* care about?
- Which of the failures could a prior fix, and which only hide?
- What would you need to do before trusting an estimate as catalogue data?
- Whose decisions are built into a tolerance, a note name or a default setting?

### Source and documentation

- [`librosa` documentation](https://librosa.org/doc/latest/index.html), including the example recordings and their credits
- [`numpy`](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [`matplotlib`](https://matplotlib.org/stable/users/explain/quick_start.html)

Sounds in Parts 1 to 4 are generated by code. Recordings in Part 5 are downloaded by `librosa` from its own collection: they are not included in this repository.

End

---